# BongoReason — The Script-Gap Experiment**What this notebook answers:** does a Bangla math model give the *same* answer when aproblem's numerals are written in Bengali digits (২৪৩) versus Arabic digits (243)?This is a falsification test, not a demo. The project's surviving contribution dependsentirely on the answer:| Cross-script agreement | Verdict ||---|---|| ≥ 95% | Models are already consistent. The contribution does not survive — re-scope. || 85–95% | A gap exists but is thin. Viable only if accuracy also differs materially. || < 85% | The gap is real. This is the paper's opening figure. |**Runtime:** set to **T4 GPU** (Runtime → Change runtime type → T4 GPU).**Time:** ~10 min for steps 1–5, then 1–3 hours for the full sweep.Run the cells in order. Steps 1–5 are cheap and catch problems early; do not skip to step 6.

## Step 0 — Check the runtimeIf this prints `No GPU`, fix the runtime type before continuing.

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
      or "No GPU — set Runtime > Change runtime type > T4 GPU")
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())

## Step 1 — Get the code and dependenciesReplace `REPO_URL` with your GitHub repository once you have created it.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/BongoReason.git"  # <-- edit this

import os
if not os.path.exists("/content/BongoReason"):
    !git clone -q $REPO_URL /content/BongoReason
%cd /content/BongoReason
!pip install -q transformers accelerate pyarrow sympy huggingface_hub
print("\nrepo ready")

### Optional but recommended: persist results to DriveColab disconnects. The eval runner resumes from whatever is already written, so putting`results/` on Drive means a disconnect costs you one batch instead of the whole run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
DRIVE_RESULTS = "/content/drive/MyDrive/BongoReason/results"
os.makedirs(DRIVE_RESULTS, exist_ok=True)
if os.path.islink("results") or os.path.exists("results"):
    shutil.rmtree("results", ignore_errors=True)
os.symlink(DRIVE_RESULTS, "results")
print("results/ ->", DRIVE_RESULTS)

## Step 2 — Fetch the evaluation benchmarksBanglaMATH and Bn-MGSM download automatically. BenNumEval is gated and will fail with 403 —that is expected and harmless, it is a secondary benchmark.**GSM-Plus-BN must be downloaded by hand** (Mendeley has no API):1. Open <https://data.mendeley.com/datasets/74dscnmrhv/3>2. Download the CSV3. Run the upload cell below and select it

In [ ]:
!python scripts/00_fetch_eval.py

### Upload GSM-Plus-BNSkip this cell if the file is already present.

In [ ]:
import os, glob
os.makedirs("dataset/eval/gsm_plus_bn", exist_ok=True)

if not glob.glob("dataset/eval/gsm_plus_bn/*.csv"):
    from google.colab import files
    up = files.upload()
    for name in up:
        os.rename(name, f"dataset/eval/gsm_plus_bn/{name}")
        print("placed", name)
else:
    print("already present:", os.listdir("dataset/eval/gsm_plus_bn"))

## Step 3 — Build the dual-script evaluation setsEmits every problem twice, once per numeral script. Problems whose two variants areidentical (no digits in the question) are excluded from the metric — they cannot disagree,so counting them would inflate agreement toward 100%.

In [ ]:
!python scripts/06_build_eval_sets.py

## Step 4 — Tokenization analysisDo this **before** the expensive run. It is nearly free and it determines how you mustframe any gap you find.If Bengali digits cost ≥2× the tokens of Arabic digits, an accuracy gap is confounded withsequence length and has to be reported that way. If fertility is roughly equal, a gapcannot be dismissed as a tokenizer artifact — which materially strengthens the paper.

In [ ]:
!python scripts/analyze_tokenization.py --model Qwen/Qwen3-0.6B

## Step 5 — Smoke test (do not skip)Twenty problems. The point is to confirm the model actually emits a parseable answerbefore you spend two hours discovering it does not.Check the printed extraction summary: if `extraction_method` is mostly `None`, the model isnot following the output format and your accuracy number would be measuring formatcompliance rather than reasoning. Fix the prompt in `scripts/run_eval.py` before continuing.

In [ ]:
!python scripts/run_eval.py --model Qwen/Qwen3-0.6B --benchmark bn_mgsm --limit 20 \
    --batch-size 8 --save-outputs

In [ ]:
# Inspect what the model actually produced
import json, collections
rows = [json.loads(l) for l in open("results/Qwen__Qwen3-0.6B/bn_mgsm.jsonl", encoding="utf-8")]

methods = collections.Counter(r["extraction_method"] for r in rows)
print("extraction methods:", dict(methods))
parsed = sum(1 for r in rows if r["extraction_method"])
print(f"parseable answers: {parsed}/{len(rows)} ({100*parsed/len(rows):.0f}%)")
if parsed / max(len(rows), 1) < 0.5:
    print("\n*** WARNING: fewer than half the outputs are parseable.")
    print("*** Fix the prompt before the full run, or you are measuring format compliance.")

print("\n--- one sample generation ---")
sample = next((r for r in rows if r.get("output")), rows[0])
print("gold:", sample["gold_answer"], "| predicted:", sample["predicted_raw"],
      "| correct:", sample["correct"])
print((sample.get("output") or "")[:700])

## Step 6 — Full run: the base modelQwen3-0.6B untuned, all three benchmarks, both numeral scripts. ~9,993 problems × 2 scripts.If the session drops, just rerun this cell — completed items are skipped.Lower `--batch-size` to 16 if you hit out-of-memory.

In [ ]:
!python scripts/run_eval.py --model Qwen/Qwen3-0.6B --benchmark all --batch-size 32

## Step 7 — Full run: the published competitorGanitLLM-0.6B is the real bar — an ACL 2026 Findings model on the same Qwen3-0.6B base.This matters as much as step 6. If the base model is script-inconsistent but GanitLLMalready is not, the problem is solved and you need to know that now rather than aftertraining your own model.

In [ ]:
!python scripts/run_eval.py --model dipta007/GanitLLM-0.6B --benchmark all --batch-size 32

## Step 8 — Compare the two models side by side

In [ ]:
import json, os

MODELS = ["Qwen/Qwen3-0.6B", "dipta007/GanitLLM-0.6B"]
rows = []
for m in MODELS:
    path = f"results/{m.replace('/', '__')}/summary.json"
    if not os.path.exists(path):
        print(f"missing {path} — run the cell for {m}")
        continue
    data = json.load(open(path, encoding="utf-8"))
    for bench, s in data["results"].items():
        rows.append((m.split("/")[-1], bench, s))

print(f"{'model':<18}{'benchmark':<14}{'n':>7}{'acc(ar)':>10}{'acc(bn)':>10}"
      f"{'gap':>8}{'agree':>9}{'flip':>8}")
print("-" * 84)
for model, bench, s in rows:
    print(f"{model:<18}{bench:<14}{s['n_paired']:>7,}"
          f"{100*s['accuracy_ar']:>9.1f}%{100*s['accuracy_bn']:>9.1f}%"
          f"{100*s['accuracy_gap']:>+7.1f}%{100*s['answer_agreement']:>8.1f}%"
          f"{100*s['correctness_flip']:>7.1f}%")

for model in {r[0] for r in rows}:
    subset = [s for m, b, s in rows if m == model]
    n = sum(s["n_paired"] for s in subset)
    agree = sum(s["answer_agreement"] * s["n_paired"] for s in subset) / n
    print(f"\n{model}: weighted agreement {100*agree:.1f}%  ->", end=" ")
    print("contribution does not survive" if agree >= 0.95
          else "thin, check accuracy gap" if agree >= 0.85
          else "REAL GAP — this is the paper")

## Step 9 — What the numbers mean- **agree** — same final answer under both scripts, correct or not. The headline number.  Two identical *wrong* answers still agree; this measures consistency, not accuracy.- **flip** — correct under one script, wrong under the other. The cleanest evidence of  script sensitivity, and the most quotable single statistic.- **gap** — accuracy difference. Positive means Arabic digits are easier for the model.Bn-MGSM is 98% Arabic-native and GSM-Plus-BN is 98% Bengali-native, so they probe oppositedirections. A one-sided effect (degradation only when converting *to* Bengali, say) would beas interesting as a symmetric one, and worth reporting separately rather than pooling.### If the gap is realNext experiment is SFT with script augmentation — no RL needed, two LoRA runs:- train split: `dataset/splits/sft_train.json` (20,555 records)- augmentation source: `dataset/script_pairs/script_pairs_v0.4.json` (22,469 pairs)Train once without augmentation, once with, hold everything else fixed, and re-run thisnotebook against both checkpoints. That contrast is the paper's core result.### If it is notRe-scope before building anything. See `docs/novelty_statement.md`, which lists thealternatives honestly.